# ML Exam Notes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

df = pd.read_csv('file.csv')

# Load data and explore it

In [ ]:
print('Shape:', df.shape)   # (rows, columns)
df.head()

## Clean Data

In [ ]:
print('NaN per column:\n', df.isnull().sum())

## Target Column Distribuition

In [ ]:
df[target_column].value_counts().plot(kind='bar', color=['steelblue', 'salmon'])
plt.xticks(rotation=0)
plt.ylabel('Count')
plt.title(f'Class Distribution — {target_column}')
plt.show()

## Feature Distribuition

In [ ]:
features = ['Col1', 'Col2', 'Col3', 'Col4']

fig, axes = plt.subplots(1, len(features), figsize=(4 * len(features), 5))
for ax, feat in zip(axes, features):
    df.boxplot(column=feat, by=target_column, ax=ax)
    ax.set_title(feat)
    ax.set_xlabel(target_column)
plt.suptitle('')
plt.tight_layout()
plt.show()

# df.groupby(target_column)[features].mean().plot(kind='bar')

# Prep The Data

In [ ]:
# --- Drop irrelevant columns ---
df = df.drop(columns=["Col1", "Col2"])

# --- Fix zero-encoded missing values ---
zero_cols = ['Col1', 'Col2']
df[zero_cols] = df[zero_cols].replace(0, pd.NA)
df[zero_cols] = df[zero_cols].fillna(df[zero_cols].median())
print('Zeros remaining:', (df[zero_cols] == 0).sum())

## Feature Scaling

Apply **StandardScaler** (mean=0, std=1). **Required** for KNN and Logistic Regression (distance/gradient sensitive). Not strictly needed for Random Forest but never hurts.

> `fit_transform` on train only → `transform` on val/test (avoid data leakage).

In [ ]:
X = df.drop(columns=TARGET)
y = df[TARGET]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Means (≈0):', X_scaled.mean(axis=0).round(2))
print('Stds  (≈1):', X_scaled.std(axis=0).round(2))

## Train / Val / Test Split

Two-step split. Always use `stratify=y` to preserve the class ratio in each split.

In [ ]:
# Use X_scaled/y if you scaled; otherwise use X/y directly.
# If the exam splits the df itself (no separate scaling step), swap X_scaled → df and y → df[TARGET].

# Step 1: hold out 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=SEED
)
# Step 2: split remaining 80% → 75% train + 25% val  =  60% / 20% overall
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=SEED
)
print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

# Model Training with GridSearchCV

**Pattern for every model:**
1. Define `param_grid`
2. `GridSearchCV(model, param_grid, cv=5, scoring='f1', n_jobs=-1)`
3. `.fit(X_train, y_train)`
4. Print best params + best CV F1

## Logistic Regression

**Justification:** Simple, interpretable binary baseline. Works well when features show roughly linear separation from the target. Fast to train, provides probability estimates. `C` controls regularisation strength (smaller C = more regularisation).

In [ ]:
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear']
}

lr_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=SEED),
    param_grid_lr, cv=5, scoring='f1', n_jobs=-1
)
lr_search.fit(X_train, y_train)

print('Best params:', lr_search.best_params_)
print('Best CV F1: ', round(lr_search.best_score_, 4))

## K-Nearest Neighbours (KNN)

**Justification:** Non-parametric, no distribution assumptions. Classifies by majority vote among the K closest training points. Requires scaled features (distance-sensitive). Good when the decision boundary is complex/non-linear. Can struggle with class imbalance and high dimensions.

In [ ]:
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance']
}

knn_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn, cv=5, scoring='f1', n_jobs=-1
)
knn_search.fit(X_train, y_train)

print('Best params:', knn_search.best_params_)
print('Best CV F1: ', round(knn_search.best_score_, 4))

## Random Forest

**Justification:** Ensemble of decision trees trained on random data/feature subsets (bagging). Handles non-linear relationships, robust to outliers, doesn't need scaling, and provides feature importances. Reduces overfitting vs a single tree. `n_estimators` = number of trees, `max_depth` = how deep each tree grows.

In [ ]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20]
}

rf_search = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_grid_rf, cv=5, scoring='f1', n_jobs=-1
)
rf_search.fit(X_train, y_train)

print('Best params:', rf_search.best_params_)
print('Best CV F1: ', round(rf_search.best_score_, 4))

# Model Comparison on Validation Set

## Evaluate All Three Models on Validation Set

In [ ]:
models = {
    'Logistic Regression': lr_search.best_estimator_,
    'KNN':                 knn_search.best_estimator_,
    'Random Forest':       rf_search.best_estimator_,
}

results = []
for name, model in models.items():
    y_pred = model.predict(X_val)
    results.append({
        'Model':    name,
        'Accuracy': round(accuracy_score(y_val, y_pred), 4),
        'F1 Score': round(f1_score(y_val, y_pred), 4),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df)

# Final Evaluation on Test Set

## Retrain on Train + Validation Combined

In [ ]:
# Combine train + val, then retrain the chosen model
X_trainval = np.concatenate([X_train, X_val])
y_trainval  = pd.concat([y_train, y_val])

# --- swap in the correct class + best_params ---
best_search  = lr_search                         # ← lr_search / knn_search / rf_search
BestClass    = LogisticRegression                # ← LogisticRegression / KNeighborsClassifier / RandomForestClassifier
extra_kwargs = {'max_iter': 1000, 'random_state': SEED}  # ← adjust per model (KNN needs none; RF needs random_state)

final_model = BestClass(**best_search.best_params_, **extra_kwargs)
final_model.fit(X_trainval, y_trainval)
print('Final model retrained on', X_trainval.shape[0], 'samples')
print('Params used:', best_search.best_params_)

## Classification Report on Test Set

In [ ]:
y_test_pred = final_model.predict(X_test)

# target_names labels: set to match your classes (e.g. ['No', 'Yes'] or ['Class0', 'Class1'])
print(classification_report(y_test, y_test_pred, target_names=['Class 0', 'Class 1']))

## Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

labels = ['Class 0', 'Class 1'] 
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()

# Key Concepts

## When to use each model

| Model | Use when |
|---|---|
| **Logistic Regression** | Linear-ish separability, small dataset, need interpretability |
| **KNN** | Non-linear, small-medium dataset, scaled features, no distribution assumptions |
| **Random Forest** | Non-linear, handles noise, larger dataset, need feature importances |

## Metrics

- **Accuracy** → misleading with imbalanced classes
- **F1** → use as primary metric when imbalanced (balances precision & recall)
- **Precision** → "when I predict positive, am I right?" (reduces false positives)
- **Recall** → "do I catch all positives?" (reduces false negatives — critical in medical/safety contexts)
- **Macro F1** → treats all classes equally (good for imbalance diagnosis)
- **Weighted F1** → skewed by majority class size (can look good even if minority class fails)